# Colab-Only ASL Demo (Self-Contained)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip -q install -U --no-cache-dir unsloth unsloth_zoo pandas huggingface_hub sentencepiece


In [5]:
import os, re, json
from collections import Counter
from typing import List
import pandas as pd

# Unsloth/HF stability preflight (must run before importing unsloth)
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ.pop('UNSLOTH_USE_MODELSCOPE', None)
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '60')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '600')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

from huggingface_hub import login
# If model is private, uncomment:
# login()

In [6]:
MODEL_ID = 'AlexD281/asl-gemma4-e2b-q64-top50-lora'
MAX_SEQ_LENGTH = 2048

backend = None
model = None
tokenizer = None

try:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        full_finetuning=False,
    )
    FastModel.for_inference(model)
    backend = 'unsloth'
    print('loaded with unsloth:', MODEL_ID)
except Exception as e:
    print('Unsloth load failed; fallback to transformers merged checkpoint')
    print('reason:', repr(e))
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    MODEL_ID_FALLBACK = 'AlexD281/asl-gemma4-e2b-q64-top50-merged-16bit'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_FALLBACK, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID_FALLBACK,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
        trust_remote_code=True,
    )
    backend = 'transformers'
    MODEL_ID = MODEL_ID_FALLBACK
    print('loaded with transformers:', MODEL_ID_FALLBACK)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth load failed; fallback to transformers merged checkpoint
reason: RuntimeError("Unsloth: No config file found - are you sure the `model_name` is correct?\nIf you're using a model on your local device, confirm if the folder location exists.\nIf you're using a HuggingFace online model, check if it exists.")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

loaded with transformers: AlexD281/asl-gemma4-e2b-q64-top50-merged-16bit


In [7]:
import os, re, json
from collections import Counter
from typing import List
import pandas as pd

# Unsloth/HF stability preflight (must run before importing unsloth)
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ.pop('UNSLOTH_USE_MODELSCOPE', None)
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '60')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '600')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

from huggingface_hub import login
# If model is private, uncomment:
# login()

# Top-50 allowlist (edit as needed to your exact canonical list)
ALLOWLIST = [
'hello','thanks','yes','no','please','sorry','help','eat','drink','water',
'bathroom','name','what','where','when','why','who','how','good','bad',
'happy','sad','angry','love','friend','family','mother','father','brother','sister',
'school','work','home','go','come','stop','wait','finished','again','more',
'today','tomorrow','yesterday','morning','night','time','know','understand','want','need'
]
ALLOWSET = set(ALLOWLIST)
ALIAS = {
  'thankyou':'thanks', 'thx':'thanks', 'ok':'yes', 'okay':'yes', 'toilet':'bathroom'
}

def normalize_gloss(x: str) -> str:
    x = (x or '').strip().lower()
    x = re.sub(r'[^a-z]+', '', x)
    return ALIAS.get(x, x)

def build_prompt(inp: str, allowlist: List[str]) -> str:
    return (
      'You are an ASL gloss decoder.\n'
      'Return exactly ONE token from this allowlist.\n'
      f"ALLOWLIST: {', '.join(allowlist)}\n"
      'If uncertain, return unknown.\n\n'
      f'INPUT: {inp}\n'
      'OUTPUT:'
    )

def raw_generate(prompt: str, max_new_tokens=4, temperature=0.0) -> str:
    if backend == 'unsloth':
        inputs = tokenizer(text=prompt, return_tensors='pt').to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = outputs[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(gen, skip_special_tokens=True).strip()
    else:
        enc = tokenizer(text=prompt, return_tensors='pt').to(model.device)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = out[0][enc['input_ids'].shape[1]:]
        return tokenizer.decode(gen, skip_special_tokens=True).strip()

def infer_single(inp: str):
    p1 = build_prompt(inp, ALLOWLIST)
    r1 = raw_generate(p1, max_new_tokens=4, temperature=0.0)
    c1 = normalize_gloss(r1)
    valid1 = c1 in ALLOWSET
    if valid1:
        return {'first_pass_raw': r1, 'retry_raw':'', 'retry_used': False, 'final_gloss': c1, 'final_valid': True}

    p2 = p1 + f'\nPrevious invalid output: {r1}\nReturn one allowlist token only.'
    r2 = raw_generate(p2, max_new_tokens=2, temperature=0.0)
    c2 = normalize_gloss(r2)
    valid2 = c2 in ALLOWSET
    return {'first_pass_raw': r1, 'retry_raw': r2, 'retry_used': True, 'final_gloss': c2 if valid2 else 'unknown', 'final_valid': valid2}

In [8]:
import json
from collections import Counter, defaultdict
from pathlib import Path
# Paths (change only if your Colab paths differ)
TEST_JSONL = Path("/content/drive/MyDrive/ASL_Top50_files/asl_unsloth_pose_train_q64_full_top50_test.jsonl")
MANIFEST_JSON = Path("/content/drive/MyDrive/ASL_Top50_files/asl_unsloth_pose_train_q64_full_top50_manifest.json")
PER_ANCHOR = 1   # use all available Top-50 test labels (1 test sample per label)
TOP_K = 50
assert TEST_JSONL.exists(), f"Missing: {TEST_JSONL}"
assert MANIFEST_JSON.exists(), f"Missing: {MANIFEST_JSON}"

# Load canonical Top-50 labels from manifest (overrides placeholder allowlist)
manifest = json.loads(MANIFEST_JSON.read_text(encoding="utf-8"))
ALLOWLIST = [normalize_gloss(x) for x in manifest.get("labels", [])]
ALLOWSET = set(ALLOWLIST)
print(f"Loaded canonical labels from manifest: {len(ALLOWLIST)}")

# 1) load test records
records = []
with TEST_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))
# 2) pick top-10 anchors by frequency in test set
freq = Counter()
for r in records:
    g = normalize_gloss(str(r.get("output", "")))
    if g in ALLOWSET:
        freq[g] += 1
anchor_glosses = [g for g, _ in freq.most_common(TOP_K)]
print("Anchors:", anchor_glosses)
# 3) build DEMO_ROWS with 3 samples per anchor (deterministic file order)
picked = defaultdict(int)
demo_rows = []
for r in records:
    expected = normalize_gloss(str(r.get("output", "")))
    if expected not in anchor_glosses:
        continue
    if picked[expected] >= PER_ANCHOR:
        continue
    demo_rows.append({
        "sample_id": str(r.get("sample_id") or r.get("id") or f"row-{len(demo_rows)+1}"),
        "expected_gloss": expected,
        "input": str(r.get("input", "")),
    })
    picked[expected] += 1
    if all(picked[g] >= PER_ANCHOR for g in anchor_glosses):
        break
# 4) quick sanity checks
missing = [g for g in anchor_glosses if picked[g] < PER_ANCHOR]
if missing:
          raise RuntimeError(f"Not enough samples for anchors: {missing}. Counts={dict(picked)}")
DEMO_ROWS = demo_rows
print("Built DEMO_ROWS:", len(DEMO_ROWS))
print("Per-anchor counts:", dict(picked))
print("Example row:", DEMO_ROWS[0])

Loaded canonical labels from manifest: 50
Anchors: ['hearing', 'decide', 'black', 'center', 'clothes', 'backpack', 'hot', 'bird', 'door', 'no', 'paper', 'need', 'candy', 'deaf', 'wrong', 'book', 'fine', 'now', 'cow', 'fish', 'city', 'yes', 'like', 'blue', 'again', 'finish', 'forget', 'drink', 'birthday', 'computer', 'kiss', 'all', 'college', 'before', 'enjoy', 'dance', 'different', 'but', 'color', 'mother', 'white', 'africa', 'orange', 'coffee', 'study', 'visit', 'cook', 'chair', 'business', 'brown']
Built DEMO_ROWS: 50
Per-anchor counts: {'hearing': 1, 'decide': 1, 'black': 1, 'center': 1, 'clothes': 1, 'backpack': 1, 'hot': 1, 'bird': 1, 'door': 1, 'no': 1, 'paper': 1, 'need': 1, 'candy': 1, 'deaf': 1, 'wrong': 1, 'book': 1, 'fine': 1, 'now': 1, 'cow': 1, 'fish': 1, 'city': 1, 'yes': 1, 'like': 1, 'blue': 1, 'again': 1, 'finish': 1, 'forget': 1, 'drink': 1, 'birthday': 1, 'computer': 1, 'kiss': 1, 'all': 1, 'college': 1, 'before': 1, 'enjoy': 1, 'dance': 1, 'different': 1, 'but': 1, 

In [9]:
rows = []
for r in DEMO_ROWS:
    pred = infer_single(r['input'])
    expected = normalize_gloss(r['expected_gloss'])
    final = pred['final_gloss']
    rows.append({
      **r,
      **pred,
      'expected_gloss': expected,
      'correct': (final == expected),
    })
df = pd.DataFrame(rows)
df.head()


,sample_id,expected_gloss,input,first_pass_raw,retry_raw,retry_used,final_gloss,final_valid,correct
0,row-1,hearing,sample_id=hearing_26986\nencoding=q64_full cli...,hearing_26,,False,hearing,True,True
1,row-2,decide,sample_id=decide_15040\nencoding=q64_full clip...,sample_id=,This,True,unknown,False,False
2,row-3,black,sample_id=black_06483\nencoding=q64_full clip=...,black_06,,False,black,True,True
3,row-4,center,sample_id=center_09719\nencoding=q64_full clip...,SROcbM,,True,unknown,False,False
4,row-5,clothes,sample_id=clothes_11305\nencoding=q64_full cli...,clothes_11,,False,clothes,True,True


In [10]:
def gatekeeper(pred_rows, per_anchor_threshold=0.70, collapse_threshold=0.40, min_n=1, max_n=1):
    anchors = sorted(set(r['expected_gloss'] for r in pred_rows))

    def summarize(use_final=True):
        per = []
        reasons = []
        valid_preds = []

        for g in anchors:
            grp = [r for r in pred_rows if r['expected_gloss']==g]
            n = len(grp)
            if n < min_n or n > max_n:
                reasons.append(f'anchor_support_out_of_range:{g}:{n}')
            corr = 0
            for r in grp:
                pred = r['final_gloss'] if use_final else normalize_gloss(r['first_pass_raw'])
                if pred in ALLOWSET:
                    valid_preds.append(pred)
                if pred == g:
                    corr += 1
            acc = (corr / n) if n else 0.0
            ok = acc >= per_anchor_threshold
            if not ok:
                reasons.append(f'anchor_below_threshold:{g}:{acc:.3f}')
            per.append({'gloss':g,'support':n,'correct':corr,'accuracy':acc,'pass':ok})

        collapse = {'mode_gloss':None,'ratio':0.0,'hard_fail':False}
        if valid_preds:
            c = Counter(valid_preds)
            m, k = c.most_common(1)[0]
            ratio = k / len(valid_preds)
            hard = ratio > collapse_threshold
            if hard:
                reasons.append(f'collapse_hard_fail:{m}:{ratio:.3f}')
            collapse = {'mode_gloss':m,'ratio':ratio,'hard_fail':hard}

        overall = (len(reasons)==0)
        return {'pass': overall, 'reasons': reasons, 'per_anchor': per, 'collapse': collapse}

    first = summarize(use_final=False)
    final = summarize(use_final=True)
    return {
      'gate_version':'colab_gatekeeper_v1_notebook',
      'first_pass': first,
      'final_after_retry': final,
      'retry_effect': {'decision_changed': first['pass'] != final['pass'], 'from': first['pass'], 'to': final['pass']}
    }

report = gatekeeper(rows)
print('FINAL_GATE:', 'PASS' if report['final_after_retry']['pass'] else 'FAIL')
print('FIRST_PASS:', 'PASS' if report['first_pass']['pass'] else 'FAIL')
print('RETRY_CHANGED_DECISION:', report['retry_effect']['decision_changed'])
if report['final_after_retry']['reasons']:
    print('FAIL_REASONS:')
    for x in report['final_after_retry']['reasons']:
        print('-', x)


FINAL_GATE: FAIL
FIRST_PASS: FAIL
RETRY_CHANGED_DECISION: False
FAIL_REASONS:
- anchor_below_threshold:again:0.000
- anchor_below_threshold:all:0.000
- anchor_below_threshold:backpack:0.000
- anchor_below_threshold:before:0.000
- anchor_below_threshold:bird:0.000
- anchor_below_threshold:birthday:0.000
- anchor_below_threshold:book:0.000
- anchor_below_threshold:business:0.000
- anchor_below_threshold:but:0.000
- anchor_below_threshold:center:0.000
- anchor_below_threshold:coffee:0.000
- anchor_below_threshold:computer:0.000
- anchor_below_threshold:cow:0.000
- anchor_below_threshold:decide:0.000
- anchor_below_threshold:door:0.000
- anchor_below_threshold:drink:0.000
- anchor_below_threshold:fine:0.000
- anchor_below_threshold:fish:0.000
- anchor_below_threshold:no:0.000
- anchor_below_threshold:now:0.000
- anchor_below_threshold:paper:0.000
- anchor_below_threshold:wrong:0.000
- anchor_below_threshold:yes:0.000


In [11]:
pd.DataFrame(report['final_after_retry']['per_anchor'])


,gloss,support,correct,accuracy,pass
0,africa,1,1,1.0,True
1,again,1,0,0.0,False
2,all,1,0,0.0,False
3,backpack,1,0,0.0,False
4,before,1,0,0.0,False
5,bird,1,0,0.0,False
6,birthday,1,0,0.0,False
7,black,1,1,1.0,True
8,blue,1,1,1.0,True
9,book,1,0,0.0,False


In [12]:
from pathlib import Path
OUT = Path('/content/asl_colab_demo_outputs')
OUT.mkdir(exist_ok=True, parents=True)
df.to_csv(OUT / 'predictions.csv', index=False)
(OUT / 'gatekeeper.json').write_text(json.dumps(report, indent=2))
print('wrote', OUT)


wrote /content/asl_colab_demo_outputs


In [15]:
total = len(df)
correct = int(df["correct"].sum()) if total else 0
accuracy = (correct / total) if total else 0.0
valid = int(df["final_valid"].sum()) if total else 0
valid_rate = (valid / total) if total else 0.0
unknown = int((df["final_gloss"] == "unknown").sum()) if total else 0
unknown_rate = (unknown / total) if total else 0.0
acc_when_valid = (correct / valid) if valid else 0.0

print("=== OVERALL SCORE ===")
print(f"Samples: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Valid predictions: {valid}/{total} ({valid_rate*100:.2f}%)")
print(f"Unknown predictions: {unknown}/{total} ({unknown_rate*100:.2f}%)")

=== OVERALL SCORE ===
Samples: 50
Correct: 27
Accuracy: 0.5400 (54.00%)
Valid predictions: 27/50 (54.00%)
Unknown predictions: 23/50 (46.00%)
